# 🗺️ TFG — Grafo de Sensores de Tráfico Madrid

Construye un grafo espacial de los 1069 sensores usando sus coordenadas GPS.

## Uso en el pipeline de predicción
```
Modelo DL  →  predice: intensidad_trafico, ocupacion, carga
                 ↓
          estimar_vmed(intensidad, ocupacion, carga)  →  vmed_estimada [km/h]
                 ↓
          grafo[sensor_A → sensor_B].weight = distancia [m]
                 ↓
          tiempo_viaje = distancia / vmed_estimada  [segundos]
                 ↓
          Dijkstra → ruta óptima + ETA
```

## Criterio de conexión
- **Mínimo 30 m**: excluye el sensor del carril contrario en el mismo punto
- **Sin límite máximo**: el siguiente sensor puede estar a varios km
- **k = 3 vecinos** válidos más cercanos por sensor (sin importar distancia)


## 1. Imports

In [19]:
import numpy as np
import pandas as pd
import networkx as nx
import folium
from pathlib import Path
from sklearn.neighbors import BallTree

DATA_PATH   = Path('data/Trafico_MODELOS2.csv')
OUTPUT_DIR  = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# Parámetros del grafo
MIN_DIST_M  = 30    # excluye sensor del carril contrario (<30 m)
K_VECINOS   = 3     # k vecinos válidos más cercanos (sin límite de distancia)
R_EARTH     = 6_371_000  # radio Tierra en metros

# Candidatos a consultar: k + buffer para absorber los que queden
# por debajo del mínimo (carriles contrarios)
K_QUERY = K_VECINOS + 15

print('Librerías OK')


Librerías OK


## 2. Extraer metadatos de sensores
Se leen solo las columnas necesarias — mucho más rápido que cargar el CSV completo.

In [20]:
# Leer solo las columnas de metadatos (5 de 22) — rápido incluso en 37.5M filas
META_COLS = ['id', 'nombre', 'longitud', 'latitud', 'tipo_elem']

frames = []
seen   = set()

for chunk in pd.read_csv(DATA_PATH, usecols=META_COLS, chunksize=500_000, low_memory=False):
    nuevos = chunk[~chunk['id'].isin(seen)].drop_duplicates('id')
    if len(nuevos):
        frames.append(nuevos)
        seen.update(nuevos['id'].tolist())
    if len(seen) >= 1069:   # tenemos todos los sensores únicos
        break

sensors = (
    pd.concat(frames, ignore_index=True)
    .drop_duplicates('id')
    .dropna(subset=['latitud', 'longitud'])
    .reset_index(drop=True)
)

sensors['latitud']  = pd.to_numeric(sensors['latitud'],  errors='coerce')
sensors['longitud'] = pd.to_numeric(sensors['longitud'], errors='coerce')
sensors = sensors.dropna(subset=['latitud','longitud']).reset_index(drop=True)

print(f'Sensores totales : {len(sensors)}')
print(f'Tipos:\n{sensors["tipo_elem"].value_counts().to_string()}')
print(f'\nEjemplo nombres M30:')
print(sensors[sensors['tipo_elem']=='M30'][['id','nombre','latitud','longitud']].to_string())

Sensores totales : 1069
Tipos:
tipo_elem
URB    985
M30     84

Ejemplo nombres M30:
        id   nombre    latitud  longitud
25    3494  PM22971  40.482875 -3.702472
26    3510  PM12961  40.482395 -3.703498
31    3521  PM30001  40.492293 -3.670773
33    3560  PM22851  40.479178 -3.715946
42    3598  PM22721  40.472879 -3.728989
43    3599  PM22781  40.475230 -3.722513
56    3706  PM43022  40.493856 -3.700040
76    3797  PM40005  40.491700 -3.671644
77    3801  PM12781  40.475156 -3.722373
78    3821  PM20025  40.481995 -3.673396
79    3822  PM20026  40.481897 -3.673532
80    3823  PM20041  40.475691 -3.674138
719   6639  PM40001  40.491676 -3.671563
720   6640  PM10013  40.481200 -3.674087
721   6641  PM10021  40.480878 -3.674043
722   6642  PM10091  40.476038 -3.674395
723   6643  PM10092  40.475980 -3.674579
724   6644  PM10141  40.470874 -3.671757
725   6645  PM10142  40.470834 -3.671843
726   6646  PM10211  40.466456 -3.668430
727   6647  PM10212  40.466414 -3.668638
728   6648  P

## 3. Construcción del grafo espacial

### Lógica del filtro de carril contrario
En una calzada doble (M30, autovías), los sensores de sentido contrario están
físicamente en el mismo punto o a pocos metros. El umbral `MIN_DIST_M = 30`
elimina esas conexiones.

Para cada sensor se consultan `K_VECINOS + 15` candidatos y se toman los
primeros `K_VECINOS` que superen el mínimo — **sin ningún límite de distancia
máxima**, por lo que aunque el siguiente sensor válido esté a 5 km se conecta igualmente.


In [ ]:
coords_rad = np.radians(sensors[['latitud', 'longitud']].values)
tree = BallTree(coords_rad, metric='haversine')

dist_rad, idx = tree.query(coords_rad, k=K_QUERY + 1)
dist_m = dist_rad * R_EARTH

G = nx.DiGraph()

for _, row in sensors.iterrows():
    G.add_node(
        row['id'],
        nombre = str(row['nombre']),
        lat    = float(row['latitud']),
        lon    = float(row['longitud']),
    )

aristas_descartadas  = 0
sensores_sin_vecinos = []

for i in range(len(sensors)):
    id_origen       = sensors.iloc[i]['id']
    vecinos_validos = 0
    for d, j in zip(dist_m[i][1:], idx[i][1:]):
        if vecinos_validos >= K_VECINOS:
            break
        id_destino = sensors.iloc[j]['id']
        if d < MIN_DIST_M:
            aristas_descartadas += 1
        else:
            G.add_edge(id_origen, id_destino, distancia_m=round(d, 1))
            vecinos_validos += 1
    if vecinos_validos == 0:
        sensores_sin_vecinos.append(id_origen)

# ── Color por sentido ────────────────────────────────────────────────────────
# Sensores dentro de MIN_DIST_M entre sí son el mismo punto en sentidos contrarios.
# Al par le asignamos azul (sentido A) y rojo (sentido B).
# Sensores sin pareja cercana quedan en naranja (calle de un solo sentido, etc.)
sensor_color = {}
procesados   = set()

for i in range(len(sensors)):
    id_a = sensors.iloc[i]['id']
    if id_a in procesados:
        continue
    for d, j in zip(dist_m[i][1:], idx[i][1:]):
        if d >= MIN_DIST_M:
            break
        id_b = sensors.iloc[j]['id']
        if id_b not in procesados:
            sensor_color[id_a] = '#1565C0'   # azul oscuro — sentido A
            sensor_color[id_b] = '#B71C1C'   # rojo oscuro  — sentido B
            procesados.update([id_a, id_b])
            break

for _, row in sensors.iterrows():
    if row['id'] not in sensor_color:
        sensor_color[row['id']] = '#E65100'  # naranja — sentido único

n_azul    = sum(1 for c in sensor_color.values() if c == '#1565C0')
n_rojo    = sum(1 for c in sensor_color.values() if c == '#B71C1C')
n_naranja = sum(1 for c in sensor_color.values() if c == '#E65100')

print(f'Nodos    : {G.number_of_nodes()}')
print(f'Aristas  : {G.number_of_edges()} (dirigidas)')
print(f'Candidatos descartados (<{MIN_DIST_M} m): {aristas_descartadas}')
print(f'Componentes débilmente conexas: {nx.number_weakly_connected_components(G)}')
print(f'\nSentido A (azul)     : {n_azul}')
print(f'Sentido B (rojo)     : {n_rojo}')
print(f'Sentido único (naranja): {n_naranja}')

dists = [d['distancia_m'] for _, _, d in G.edges(data=True)]
print(f'\nDistancia media  : {np.mean(dists):.0f} m')
print(f'Distancia máxima : {max(dists):.0f} m  ({max(dists)/1000:.2f} km)')
print(f'Distancia mínima : {min(dists):.0f} m')


## 3b. Garantizar conectividad fuerte

Con K=3 aristas dirigidas por nodo puede ocurrir que un sensor no sea elegido por nadie como vecino y quede inalcanzable. Este paso añade las aristas mínimas necesarias para que desde cualquier sensor se pueda llegar a todos los demás (conectividad fuerte).

In [ ]:
def _par_cercano(G, set_a, set_b, crd, id2idx, ids, R=R_EARTH):
    """Par (u∈A, v∈B) de mínima distancia haversine usando BallTree."""
    idx_a = np.array([id2idx[n] for n in set_a])
    idx_b = np.array([id2idx[n] for n in set_b])
    ids_b = [ids[i] for i in idx_b]
    bt    = BallTree(crd[idx_b], metric='haversine')
    dists, near = bt.query(crd[idx_a], k=1)
    k = int(np.argmin(dists[:, 0]))
    return ids[idx_a[k]], ids_b[int(near[k, 0])], float(dists[k, 0] * R)

ids_g    = sensors['id'].tolist()
id2idx_g = {sid: i for i, sid in enumerate(ids_g)}
crd_g    = np.radians(sensors[['latitud', 'longitud']].values)

puentes       = 0
n_wcc_inicial = nx.number_weakly_connected_components(G)

# ── Paso 1: conectividad débil ──────────────────────────────────────────────
# Conecta componentes aislados al componente principal con aristas bidireccionales
while nx.number_weakly_connected_components(G) > 1:
    comps = sorted(nx.weakly_connected_components(G), key=len, reverse=True)
    main  = comps[0]
    otros = set().union(*comps[1:])
    u, v, d = _par_cercano(G, otros, main, crd_g, id2idx_g, ids_g)
    G.add_edge(u, v, distancia_m=round(d, 1))
    G.add_edge(v, u, distancia_m=round(d, 1))
    puentes += 2

# ── Paso 2: conectividad fuerte ─────────────────────────────────────────────
# Itera sobre la condensación (DAG de SCCs) y añade la arista mínima
# necesaria para eliminar fuentes y sumideros hasta que el grafo sea fuertemente conexo.
for _ in range(2 * G.number_of_nodes()):
    if nx.is_strongly_connected(G):
        break
    cond   = nx.condensation(G)
    main_c = max(cond.nodes(), key=lambda n: len(cond.nodes[n]['members']))
    main_m = cond.nodes[main_c]['members']
    fijo   = False

    for c in cond.nodes():
        if c == main_c:
            continue
        mbrs = cond.nodes[c]['members']
        if cond.out_degree(c) == 0:        # sumidero: atrapado, necesita salida
            u, v, d = _par_cercano(G, mbrs, main_m, crd_g, id2idx_g, ids_g)
            G.add_edge(u, v, distancia_m=round(d, 1))
            puentes += 1; fijo = True; break
        if cond.in_degree(c) == 0:         # fuente: inaccesible, necesita entrada
            u, v, d = _par_cercano(G, main_m, mbrs, crd_g, id2idx_g, ids_g)
            G.add_edge(u, v, distancia_m=round(d, 1))
            puentes += 1; fijo = True; break

    if not fijo:   # SCC intermedio sin resolver → conectar bidireccionalmente
        for c in cond.nodes():
            if c == main_c:
                continue
            mbrs = cond.nodes[c]['members']
            u, v, d = _par_cercano(G, mbrs, main_m, crd_g, id2idx_g, ids_g)
            G.add_edge(u, v, distancia_m=round(d, 1))
            G.add_edge(v, u, distancia_m=round(d, 1))
            puentes += 2; break

print(f'Componentes débiles antes  : {n_wcc_inicial}')
print(f'Aristas puente añadidas    : {puentes}')
print(f'Componentes débiles ahora  : {nx.number_weakly_connected_components(G)}')
print(f'Fuertemente conexo         : {nx.is_strongly_connected(G)}')
print(f'Nodos  : {G.number_of_nodes()}   Aristas : {G.number_of_edges()}')

## 4. Inspección de sensores descartados (carril contrario)
Muestra los pares de sensores más cercanos para verificar que el filtro es correcto.

In [22]:
# Pares de sensores a <30m — deberían ser carriles contrarios o misma ubicación
pares_cercanos = []
for i in range(len(sensors)):
    for d, j in zip(dist_m[i][1:], idx[i][1:]):
        if d < MIN_DIST_M:
            pares_cercanos.append({
                'id_A'    : sensors.iloc[i]['id'],
                'nombre_A': sensors.iloc[i]['nombre'],
                'id_B'    : sensors.iloc[j]['id'],
                'nombre_B': sensors.iloc[j]['nombre'],
                'tipo'    : sensors.iloc[i]['tipo_elem'],
                'dist_m'  : round(d, 1),
            })

df_cercanos = pd.DataFrame(pares_cercanos).drop_duplicates()
print(f'Pares a menos de {MIN_DIST_M} m: {len(df_cercanos)}')
if len(df_cercanos):
    print('\n⚠️  Revisa si alguno NO debería ser carril contrario:')
    print(df_cercanos.sort_values('dist_m').head(20).to_string(index=False))

Pares a menos de 30 m: 466

⚠️  Revisa si alguno NO debería ser carril contrario:
 id_A                                                            nombre_A  id_B                                                            nombre_B tipo  dist_m
 3659                    LÃ³pez de Hoyos - Guisona-Gran VÃ­a de Hortaleza 10741                    LÃ³pez de Hoyos - Guisona-Gran VÃ­a de Hortaleza  URB     3.0
10741                    LÃ³pez de Hoyos - Guisona-Gran VÃ­a de Hortaleza  3659                    LÃ³pez de Hoyos - Guisona-Gran VÃ­a de Hortaleza  URB     3.0
10759           Av. San Luis  - Ctra. Acceso EstaciÃ³n de Hortaleza-Lorca  6292                    Av. San Luis - Lorca-Ctra. Estacion de Hortaleza  URB     3.0
 6292                    Av. San Luis - Lorca-Ctra. Estacion de Hortaleza 10759           Av. San Luis  - Ctra. Acceso EstaciÃ³n de Hortaleza-Lorca  URB     3.0
 6111          Av. San Luis 77 O-E - Eladio LÃ³pez Vilches-Julio DÃ¡nvila 10746               Avda. San Luis - El

## 5. Guardar grafo en disco

In [23]:
# GraphML — formato estándar, importable en Gephi, QGIS, etc.
nx.write_graphml(G, OUTPUT_DIR / 'grafo_sensores.graphml')

# También guardamos la tabla de sensores
sensors.to_csv(OUTPUT_DIR / 'sensores_metadata.csv', index=False)

print(f'Guardado: {OUTPUT_DIR}/grafo_sensores.graphml')
print(f'Guardado: {OUTPUT_DIR}/sensores_metadata.csv')

Guardado: outputs/grafo_sensores.graphml
Guardado: outputs/sensores_metadata.csv


## 6. Visualización interactiva con Folium

In [ ]:
import math

def _bearing(lat1, lon1, lat2, lon2):
    lat1r, lat2r = math.radians(lat1), math.radians(lat2)
    dlon = math.radians(lon2 - lon1)
    x = math.sin(dlon) * math.cos(lat2r)
    y = math.cos(lat1r)*math.sin(lat2r) - math.sin(lat1r)*math.cos(lat2r)*math.cos(dlon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

lat_c = sensors['latitud'].mean()
lon_c = sensors['longitud'].mean()

m = folium.Map(location=[lat_c, lon_c], zoom_start=13, tiles='CartoDB positron')

# ── Aristas (PolyLine + flecha pequeña en el punto 65% del tramo) ──────────
for u, v, data in G.edges(data=True):
    nu, nv = G.nodes[u], G.nodes[v]
    p1 = (nu['lat'], nu['lon'])
    p2 = (nv['lat'], nv['lon'])

    folium.PolyLine(
        [p1, p2],
        weight=1.2, color='#78909C', opacity=0.45,
        tooltip=f"{u} → {v} | {data['distancia_m']:.0f} m",
    ).add_to(m)

    # Flecha ligera en el 65% del tramo
    mid_lat = p1[0] + 0.65 * (p2[0] - p1[0])
    mid_lon = p1[1] + 0.65 * (p2[1] - p1[1])
    brng    = _bearing(p1[0], p1[1], p2[0], p2[1])
    folium.Marker(
        location=(mid_lat, mid_lon),
        icon=folium.DivIcon(
            html=f'<div style="transform:rotate({brng}deg);'
                 f'font-size:7px;color:#37474F;line-height:1;">▲</div>',
            icon_size=(8, 8),
            icon_anchor=(4, 4),
        ),
    ).add_to(m)

# ── Nodos: azul / rojo / naranja por sentido ───────────────────────────────
for node, data in G.nodes(data=True):
    color = sensor_color.get(node, '#E65100')
    folium.CircleMarker(
        location=[data['lat'], data['lon']],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        popup=folium.Popup(
            f"<b>{node}</b><br>{data['nombre']}<br>"
            f"({data['lat']:.5f}, {data['lon']:.5f})<br>"
            f"Salidas: {G.out_degree(node)}  Entradas: {G.in_degree(node)}",
            max_width=260,
        ),
    ).add_to(m)

legend = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:10px;border-radius:8px;
     border:1px solid #ccc;font-size:13px;">
  <b>Sensores — sentido de circulación</b><br>
  <span style="color:#1565C0;font-size:16px;">●</span> Sentido A<br>
  <span style="color:#B71C1C;font-size:16px;">●</span> Sentido B<br>
  <span style="color:#E65100;font-size:16px;">●</span> Sentido único<br>
  <span style="color:#78909C;">▲</span> Dirección del flujo
</div>
"""
m.get_root().html.add_child(folium.Element(legend))

out_html = OUTPUT_DIR / 'grafo_sensores.html'
m.save(str(out_html))
print(f'Mapa guardado: {out_html}')
m


## 7. Función de tiempo de viaje

Integración con las predicciones del modelo DL.

### Estimación de vmed
El modelo predice `intensidad_trafico`, `ocupacion` y `carga`.
Cuando `vmed` no está disponible o es 0, se estima con la relación empírica
del diagrama fundamental del tráfico (modelo Greenshields simplificado):

```
vmed ≈ vf × (1 − ocupacion/100)
```
donde `vf` es la velocidad libre (configurable según tipo de vía).

In [25]:
def estimar_vmed(intensidad, ocupacion, carga, tipo_elem='URB'):
    """
    Estima velocidad media [km/h] desde las variables de estado del tráfico.
    Usa modelo Greenshields simplificado: v = vf * (1 - ocupacion/100).

    Parámetros
    ----------
    intensidad : float  veh/15min
    ocupacion  : float  % ocupación del detector
    carga      : float  % carga
    tipo_elem  : str    'M30' | 'URB'
    """
    vf = 90.0 if tipo_elem == 'M30' else 50.0   # velocidad libre [km/h]
    ocu = max(0.0, min(float(ocupacion), 100.0))
    v   = vf * (1.0 - ocu / 100.0)
    return max(5.0, v)   # mínimo 5 km/h para evitar división por cero


def tiempo_viaje_ruta(
    grafo,
    sensor_origen,
    sensor_destino,
    predicciones: dict,   # {sensor_id: {'intensidad':x,'ocupacion':y,'carga':z}}
):
    """
    Calcula la ruta óptima y el tiempo estimado de viaje usando Dijkstra.

    Parámetros
    ----------
    grafo           : nx.Graph  con atributo 'distancia_m' en aristas
    sensor_origen   : str       id del sensor de inicio
    sensor_destino  : str       id del sensor de destino
    predicciones    : dict      predicciones del modelo para cada sensor

    Retorna
    -------
    ruta       : list[str]   secuencia de sensor_ids
    tiempo_seg : float       tiempo estimado en segundos
    distancia_m: float       distancia total en metros
    """
    # Crear copia temporal con pesos en segundos
    H = grafo.copy()
    for u, v, data in H.edges(data=True):
        pred_u = predicciones.get(u, {})
        pred_v = predicciones.get(v, {})
        tipo_u = H.nodes[u].get('tipo', 'URB')
        tipo_v = H.nodes[v].get('tipo', 'URB')

        v_u = estimar_vmed(
            pred_u.get('intensidad', 50),
            pred_u.get('ocupacion', 20),
            pred_u.get('carga', 20),
            tipo_u,
        )
        v_v = estimar_vmed(
            pred_v.get('intensidad', 50),
            pred_v.get('ocupacion', 20),
            pred_v.get('carga', 20),
            tipo_v,
        )
        v_media_ms = ((v_u + v_v) / 2) * 1000 / 3600   # km/h → m/s
        data['tiempo_s'] = data['distancia_m'] / v_media_ms

    ruta        = nx.shortest_path(H, sensor_origen, sensor_destino, weight='tiempo_s')
    tiempo_seg  = nx.shortest_path_length(H, sensor_origen, sensor_destino, weight='tiempo_s')
    distancia_m = sum(
        H[ruta[i]][ruta[i+1]]['distancia_m'] for i in range(len(ruta)-1)
    )
    return ruta, tiempo_seg, distancia_m


print('Funciones cargadas')
print('\n--- Ejemplo: estimar_vmed ---')
for ocu in [10, 30, 60, 90]:
    v = estimar_vmed(100, ocu, ocu*0.8, 'URB')
    print(f'  ocupacion={ocu}%  →  vmed_est={v:.1f} km/h')

Funciones cargadas

--- Ejemplo: estimar_vmed ---
  ocupacion=10%  →  vmed_est=45.0 km/h
  ocupacion=30%  →  vmed_est=35.0 km/h
  ocupacion=60%  →  vmed_est=20.0 km/h
  ocupacion=90%  →  vmed_est=5.0 km/h


## 8. Ejemplo de uso con predicciones ficticias

In [26]:
# Ejemplo con dos sensores reales del grafo
nodos = list(G.nodes())
origen  = nodos[0]
destino = nodos[min(10, len(nodos)-1)]   # 10 saltos en el grafo

# Predicciones ficticias (en producción vendrán del modelo DL)
predicciones_ejemplo = {
    n: {
        'intensidad': np.random.uniform(20, 150),
        'ocupacion' : np.random.uniform(10, 70),
        'carga'     : np.random.uniform(10, 70),
    }
    for n in G.nodes()
}

try:
    ruta, t_seg, dist_m = tiempo_viaje_ruta(
        G, origen, destino, predicciones_ejemplo
    )
    print(f'Origen  : {origen}  →  {G.nodes[origen]["nombre"]}')
    print(f'Destino : {destino}  →  {G.nodes[destino]["nombre"]}')
    print(f'\nRuta ({len(ruta)} sensores): {" → ".join(str(s) for s in ruta)}')
    print(f'Distancia total : {dist_m:.0f} m  ({dist_m/1000:.2f} km)')
    print(f'Tiempo estimado : {t_seg:.0f} s  ({t_seg/60:.1f} min)')
except nx.NetworkXNoPath:
    print(f'No hay camino entre {origen} y {destino} en el grafo.')
    print('Prueba con otros nodos o aumenta MAX_DIST_M.')

No hay camino entre 3402 y 3440 en el grafo.
Prueba con otros nodos o aumenta MAX_DIST_M.


## 9. Visualizar una ruta concreta en el mapa

In [27]:
# Calcular la ruta (por si no se ejecutó la celda anterior o falló)
nodos = list(G.nodes())
origen  = nodos[0]
destino = nodos[min(10, len(nodos)-1)]

predicciones_ejemplo = {
    n: {
        'intensidad': np.random.uniform(20, 150),
        'ocupacion' : np.random.uniform(10, 70),
        'carga'     : np.random.uniform(10, 70),
    }
    for n in G.nodes()
}

try:
    ruta, t_seg, dist_m = tiempo_viaje_ruta(G, origen, destino, predicciones_ejemplo)
except nx.NetworkXNoPath:
    print(f'No hay camino entre {origen} y {destino}. Buscando par alternativo...')
    ruta = t_seg = dist_m = None
    for i, src in enumerate(nodos[:20]):
        for dst in nodos[i+5:i+25]:
            try:
                ruta, t_seg, dist_m = tiempo_viaje_ruta(G, src, dst, predicciones_ejemplo)
                origen, destino = src, dst
                break
            except nx.NetworkXNoPath:
                continue
        if ruta is not None:
            break

if ruta is None:
    print('No se encontró ninguna ruta viable en el grafo.')
else:
    print(f'Origen  : {origen}')
    print(f'Destino : {destino}')
    print(f'Ruta    : {len(ruta)} sensores | {dist_m:.0f} m | {t_seg/60:.1f} min')

    m_ruta = folium.Map(location=[lat_c, lon_c], zoom_start=13, tiles='CartoDB positron')

    # Todas las aristas en gris claro
    for u, v, data in G.edges(data=True):
        nu, nv = G.nodes[u], G.nodes[v]
        folium.PolyLine(
            [(nu['lat'], nu['lon']), (nv['lat'], nv['lon'])],
            weight=1, color='#BBBBBB', opacity=0.3
        ).add_to(m_ruta)

    # Ruta óptima en rojo
    ruta_coords = [(G.nodes[s]['lat'], G.nodes[s]['lon']) for s in ruta]
    folium.PolyLine(
        ruta_coords, weight=4, color='#E53935', opacity=0.9,
        tooltip=f'Ruta: {dist_m:.0f} m | {t_seg/60:.1f} min'
    ).add_to(m_ruta)

    # Nodos de la ruta coloreados: verde=inicio, rojo=fin, naranja=intermedios
    for i, s in enumerate(ruta):
        nd = G.nodes[s]
        color = '#4CAF50' if i == 0 else ('#F44336' if i == len(ruta)-1 else '#FF9800')
        folium.CircleMarker(
            [nd['lat'], nd['lon']], radius=7, color=color,
            fill=True, fill_opacity=0.9,
            popup=f"{s}: {nd['nombre']}"
        ).add_to(m_ruta)

    out_ruta = OUTPUT_DIR / 'ruta_ejemplo.html'
    m_ruta.save(str(out_ruta))
    print(f'Mapa guardado: {out_ruta}')
    display(m_ruta)


No hay camino entre 3402 y 3440. Buscando par alternativo...
Origen  : 3408
Destino : 3485
Ruta    : 13 sensores | 1388 m | 2.6 min
Mapa guardado: outputs/ruta_ejemplo.html


## 10. Subgrafo filtrado — solo sensores en calzada doble (azul + rojo)

Elimina los sensores **naranja** (sentido único, sin pareja a < 30 m) y construye
un grafo limpio solo con los pares de carriles contrarios.

Útil para análisis de bidireccionalidad o para reducir el grafo a vías con
información de ambos sentidos.

In [ ]:
# Nodos que son azul o rojo (tienen pareja de carril contrario a < 30 m)
nodos_par = {
    node for node, color in sensor_color.items()
    if color in ('#1565C0', '#B71C1C')
}

# Subgrafo inducido: solo esos nodos y las aristas entre ellos
G_par = G.subgraph(nodos_par).copy()

n_azul_par = sum(1 for n in G_par.nodes() if sensor_color[n] == '#1565C0')
n_rojo_par = sum(1 for n in G_par.nodes() if sensor_color[n] == '#B71C1C')

print(f'Grafo original  → nodos: {G.number_of_nodes():>4}  aristas: {G.number_of_edges()}')
print(f'Eliminados (naranja/sentido único): {G.number_of_nodes() - len(nodos_par)}')
print(f'Subgrafo G_par  → nodos: {G_par.number_of_nodes():>4}  aristas: {G_par.number_of_edges()}')
print(f'  · Sentido A (azul) : {n_azul_par}')
print(f'  · Sentido B (rojo) : {n_rojo_par}')
print(f'Componentes débilmente conexas: {nx.number_weakly_connected_components(G_par)}')

In [ ]:
# Guardar el subgrafo filtrado
nx.write_graphml(G_par, OUTPUT_DIR / 'grafo_sensores_par.graphml')
print(f'Guardado: {OUTPUT_DIR}/grafo_sensores_par.graphml')

In [ ]:
lat_c = sensors['latitud'].mean()
lon_c = sensors['longitud'].mean()

m_par = folium.Map(location=[lat_c, lon_c], zoom_start=13, tiles='CartoDB positron')

# Aristas del subgrafo
for u, v, data in G_par.edges(data=True):
    nu, nv = G_par.nodes[u], G_par.nodes[v]
    p1 = (nu['lat'], nu['lon'])
    p2 = (nv['lat'], nv['lon'])

    folium.PolyLine(
        [p1, p2],
        weight=1.2, color='#78909C', opacity=0.45,
        tooltip=f"{u} → {v} | {data['distancia_m']:.0f} m",
    ).add_to(m_par)

    mid_lat = p1[0] + 0.65 * (p2[0] - p1[0])
    mid_lon = p1[1] + 0.65 * (p2[1] - p1[1])
    brng    = _bearing(p1[0], p1[1], p2[0], p2[1])
    folium.Marker(
        location=(mid_lat, mid_lon),
        icon=folium.DivIcon(
            html=f'<div style="transform:rotate({brng}deg);'
                 f'font-size:7px;color:#37474F;line-height:1;">▲</div>',
            icon_size=(8, 8), icon_anchor=(4, 4),
        ),
    ).add_to(m_par)

# Nodos: solo azul y rojo
for node, data in G_par.nodes(data=True):
    color = sensor_color[node]
    folium.CircleMarker(
        location=[data['lat'], data['lon']],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        popup=folium.Popup(
            f"<b>{node}</b><br>{data['nombre']}<br>"
            f"({data['lat']:.5f}, {data['lon']:.5f})<br>"
            f"Salidas: {G_par.out_degree(node)}  Entradas: {G_par.in_degree(node)}",
            max_width=260,
        ),
    ).add_to(m_par)

legend_par = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:10px;border-radius:8px;
     border:1px solid #ccc;font-size:13px;">
  <b>Subgrafo — calzada doble (pares)</b><br>
  <span style="color:#1565C0;font-size:16px;">●</span> Sentido A<br>
  <span style="color:#B71C1C;font-size:16px;">●</span> Sentido B<br>
  <span style="color:#78909C;">▲</span> Dirección del flujo
</div>
"""
m_par.get_root().html.add_child(folium.Element(legend_par))

out_par = OUTPUT_DIR / 'grafo_sensores_par.html'
m_par.save(str(out_par))
print(f'Mapa guardado: {out_par}')
m_par

## 11. Subgrafos dirigidos por sentido de circulación

Separa los pares identificados en §3 en dos DiGraph independientes:
- **G_A** — sensores azules (sentido A / ida)
- **G_B** — sensores rojos (sentido B / vuelta)

Cada subgrafo hereda las aristas de G que conectan sensores del **mismo sentido**
(aristas cruzadas hacia el sentido contrario se descartan). Luego se aplica el mismo
mecanismo de puentes del paso 3b para garantizar conectividad fuerte dentro de cada uno.

```
G completo  ->  inducido por nodos_A  ->  +puentes  ->  G_A (fuertemente conexo)
            ->  inducido por nodos_B  ->  +puentes  ->  G_B (fuertemente conexo)
```

In [ ]:
# ── Función reutilizable: aplica el mismo algoritmo de puentes que el paso 3b ──────────
def _hacer_fuertemente_conexo(H, nombre, crd, id2idx, ids):
    """Garantiza conectividad fuerte en el DiGraph H añadiendo las aristas mínimas."""
    puentes = 0
    n_wcc0  = nx.number_weakly_connected_components(H)

    # Paso 1: conectividad débil
    while nx.number_weakly_connected_components(H) > 1:
        comps = sorted(nx.weakly_connected_components(H), key=len, reverse=True)
        main  = comps[0]
        otros = set().union(*comps[1:])
        u, v, d = _par_cercano(H, otros, main, crd, id2idx, ids)
        H.add_edge(u, v, distancia_m=round(d, 1))
        H.add_edge(v, u, distancia_m=round(d, 1))
        puentes += 2

    # Paso 2: conectividad fuerte
    for _ in range(2 * H.number_of_nodes()):
        if nx.is_strongly_connected(H):
            break
        cond   = nx.condensation(H)
        main_c = max(cond.nodes(), key=lambda n: len(cond.nodes[n]['members']))
        main_m = cond.nodes[main_c]['members']
        fijo   = False

        for c in cond.nodes():
            if c == main_c:
                continue
            mbrs = cond.nodes[c]['members']
            if cond.out_degree(c) == 0:        # sumidero: necesita arista de salida
                u, v, d = _par_cercano(H, mbrs, main_m, crd, id2idx, ids)
                H.add_edge(u, v, distancia_m=round(d, 1))
                puentes += 1; fijo = True; break
            if cond.in_degree(c) == 0:         # fuente: necesita arista de entrada
                u, v, d = _par_cercano(H, main_m, mbrs, crd, id2idx, ids)
                H.add_edge(u, v, distancia_m=round(d, 1))
                puentes += 1; fijo = True; break

        if not fijo:   # SCC intermedio sin resolver -> bidireccional
            for c in cond.nodes():
                if c == main_c:
                    continue
                mbrs = cond.nodes[c]['members']
                u, v, d = _par_cercano(H, mbrs, main_m, crd, id2idx, ids)
                H.add_edge(u, v, distancia_m=round(d, 1))
                H.add_edge(v, u, distancia_m=round(d, 1))
                puentes += 2; break

    estado = "OK" if nx.is_strongly_connected(H) else "FALLO"
    print(f"{nombre}: WCC_inicial={n_wcc0} | puentes={puentes} | "
          f"nodos={H.number_of_nodes()} | aristas={H.number_of_edges()} | "
          f"fuerte={nx.is_strongly_connected(H)} [{estado}]")
    return H


# ── Partición de nodos por sentido ───────────────────────────────────────────
nodos_A = {n for n, c in sensor_color.items() if c == '#1565C0'}
nodos_B = {n for n, c in sensor_color.items() if c == '#B71C1C'}

print(f'Nodos sentido A (azul) : {len(nodos_A)}')
print(f'Nodos sentido B (rojo) : {len(nodos_B)}')
print()

# ── Subgrafos inducidos: solo aristas cuyos dos extremos son del mismo sentido ─
G_A = G.subgraph(nodos_A).copy()
G_B = G.subgraph(nodos_B).copy()

print(f'G_A inducido -> nodos: {G_A.number_of_nodes():>4}  '
      f'aristas: {G_A.number_of_edges():>5}  '
      f'WCC: {nx.number_weakly_connected_components(G_A)}')
print(f'G_B inducido -> nodos: {G_B.number_of_nodes():>4}  '
      f'aristas: {G_B.number_of_edges():>5}  '
      f'WCC: {nx.number_weakly_connected_components(G_B)}')
print()

# ── Garantizar conectividad fuerte en cada subgrafo ──────────────────────────
G_A = _hacer_fuertemente_conexo(G_A, 'G_A', crd_g, id2idx_g, ids_g)
G_B = _hacer_fuertemente_conexo(G_B, 'G_B', crd_g, id2idx_g, ids_g)


In [ ]:
# ── Estadísticas finales de cada subgrafo ────────────────────────────────────
sep = '-' * 62
print(sep)
print(f"{'Metrica':<30s}  {'G_A (ida)':>12s}  {'G_B (vuelta)':>12s}")
print(sep)

rows = {}
for etiqueta, H in [('G_A', G_A), ('G_B', G_B)]:
    dists    = [d['distancia_m'] for _, _, d in H.edges(data=True)]
    sccs     = list(nx.strongly_connected_components(H))
    out_degs = [H.out_degree(n) for n in H.nodes()]
    in_degs  = [H.in_degree(n)  for n in H.nodes()]
    rows[etiqueta] = {
        'Nodos'                : H.number_of_nodes(),
        'Aristas'              : H.number_of_edges(),
        'SCCs'                 : len(sccs),
        'Fuertemente conexo'   : nx.is_strongly_connected(H),
        'Grado salida medio'   : round(float(np.mean(out_degs)), 2),
        'Grado salida maximo'  : max(out_degs),
        'Grado entrada medio'  : round(float(np.mean(in_degs)), 2),
        'Grado entrada maximo' : max(in_degs),
        'Dist media (m)'       : round(float(np.mean(dists))),
        'Dist maxima (m)'      : round(max(dists)),
        'Dist minima (m)'      : round(min(dists)),
    }

for metrica in rows['G_A']:
    v_a = rows['G_A'][metrica]
    v_b = rows['G_B'][metrica]
    print(f"  {metrica:<28s}  {str(v_a):>12s}  {str(v_b):>12s}")

print(sep)


In [ ]:
nx.write_graphml(G_A, OUTPUT_DIR / 'grafo_sentido_A.graphml')
nx.write_graphml(G_B, OUTPUT_DIR / 'grafo_sentido_B.graphml')
print(f'Guardado: {OUTPUT_DIR}/grafo_sentido_A.graphml')
print(f'Guardado: {OUTPUT_DIR}/grafo_sentido_B.graphml')


In [ ]:
# Mapa superpuesto: G_A en azul y G_B en rojo.
# En calles de doble sentido los dos subgrafos deben verse paralelos
# con flechas en direcciones opuestas — eso valida que la separación es correcta.
_COLOR_A = '#1565C0'   # azul — sentido A
_COLOR_B = '#B71C1C'   # rojo  — sentido B

m_ab = folium.Map(location=[lat_c, lon_c], zoom_start=13, tiles='CartoDB positron')

for H, color, lbl in [(G_A, _COLOR_A, 'A'), (G_B, _COLOR_B, 'B')]:
    for u, v, data in H.edges(data=True):
        nu, nv = H.nodes[u], H.nodes[v]
        p1 = (nu['lat'], nu['lon'])
        p2 = (nv['lat'], nv['lon'])
        folium.PolyLine(
            [p1, p2], weight=1.5, color=color, opacity=0.55,
            tooltip=f"Sentido {lbl}: {u}→{v} | {data['distancia_m']:.0f} m",
        ).add_to(m_ab)
        # flecha en el 65% del tramo
        mid_lat = p1[0] + 0.65 * (p2[0] - p1[0])
        mid_lon = p1[1] + 0.65 * (p2[1] - p1[1])
        brng = _bearing(p1[0], p1[1], p2[0], p2[1])
        folium.Marker(
            location=(mid_lat, mid_lon),
            icon=folium.DivIcon(
                html=(f'<div style="transform:rotate({brng}deg);font-size:7px;'
                      f'color:{color};line-height:1;">▲</div>'),
                icon_size=(8, 8), icon_anchor=(4, 4),
            ),
        ).add_to(m_ab)

for H, color in [(G_A, _COLOR_A), (G_B, _COLOR_B)]:
    for node, data in H.nodes(data=True):
        folium.CircleMarker(
            location=[data['lat'], data['lon']],
            radius=4, color=color, fill=True,
            fill_color=color, fill_opacity=0.9,
            popup=folium.Popup(
                f"<b>{node}</b><br>{data['nombre']}<br>"
                f"({data['lat']:.5f}, {data['lon']:.5f})<br>"
                f"Out: {H.out_degree(node)}  In: {H.in_degree(node)}",
                max_width=240,
            ),
        ).add_to(m_ab)

legend_ab = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:10px;border-radius:8px;
     border:1px solid #ccc;font-size:13px;">
  <b>Subgrafos por sentido de circulación</b><br>
  <span style="color:#1565C0;font-size:16px;">&#9679;</span> Sentido A (ida / azul)<br>
  <span style="color:#B71C1C;font-size:16px;">&#9679;</span> Sentido B (vuelta / rojo)<br>
  <span style="font-size:10px;">&#9650; dirección del flujo</span>
</div>
"""
m_ab.get_root().html.add_child(folium.Element(legend_ab))

out_ab = OUTPUT_DIR / 'grafo_sentidos_AB.html'
m_ab.save(str(out_ab))
print(f'Mapa guardado: {out_ab}')
m_ab


## 12. Grafo OSMnx — Sensores anclados a la red vial real

Las celdas anteriores (§3–§11) construyen el grafo por **proximidad haversine**:
las aristas se añaden al sensor geográficamente más cercano, sin comprobar si hay
carretera entre ellos. Esto provoca que sensores de la M-30 separados por un largo
tramo sin sensores intermedios queden desconectados.

### Solución implementada aquí
1. **Descarga la red vial OSM** (carreteras + autovías) para el bounding box de los
   sensores mediante OSMnx.
2. **Ancla cada sensor** a su nodo OSM más cercano (`ox.nearest_nodes`).
3. **Calcula distancias viales reales** con Dijkstra sobre el grafo OSM, en lugar
   de distancia en línea recta haversine.
4. **Filtro de sensor intermedio**: conecta i→j solo si ningún sensor k cumple
   `road_dist(i,k) + road_dist(k,j) ≤ road_dist(i,j) × 1.05`
   (k estaría en la ruta i→j, es decir, es el sucesor real de i, no j).
5. **Mantiene separación A/B** usando los pares a <30 m ya identificados en §3.
6. **Conectividad fuerte** garantizada reutilizando `_hacer_fuertemente_conexo` de §11.
7. **Mapa Folium** con las polilíneas de ruta real sobre el mapa (no líneas rectas).
8. **Sobreescribe** `outputs/grafo_sentido_A.graphml` y `outputs/grafo_sentido_B.graphml`
   con los grafos basados en OSMnx.


In [ ]:
# ── Dependencias: OSMnx + verificar osmium-tool ────────────────────────────────
import importlib, subprocess, sys, math as _math

def _ensure(pkg, import_as=None):
    name = import_as or pkg
    try:
        return importlib.import_module(name)
    except ImportError:
        print(f'Instalando {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
        return importlib.import_module(name)

ox = _ensure('osmnx')
print(f'OSMnx {ox.__version__}  OK')

# osmium-tool es un binario del sistema (no paquete Python).
# Si falta, instálalo UNA VEZ en terminal y reinicia el kernel:
#   conda install -c conda-forge osmium-tool -y
try:
    subprocess.run(['osmium', '--version'], capture_output=True, check=True)
    print('osmium-tool  OK')
except FileNotFoundError:
    raise RuntimeError(
        '\n  osmium-tool no encontrado.\n'
        '  Ejecuta en terminal:  conda install -c conda-forge osmium-tool -y\n'
        '  Luego reinicia el kernel y re-ejecuta esta celda.'
    )

ox.settings.use_cache   = True
ox.settings.log_console = False


In [ ]:
# ── Carga la red vial OSM desde fichero local (sin Overpass) ─────────────────
# Paso 1: osmium filtra solo carreteras transitables del PBF → .osm.bz2
#         (solo se hace una vez; el archivo filtrado queda guardado)
# Paso 2: osmnx lee el XML comprimido y construye el DiGraph

PBF_PATH = OUTPUT_DIR / 'madrid-260505.osm.pbf'
OSM_BZ2  = OUTPUT_DIR / 'madrid-roads.osm.bz2'

if not OSM_BZ2.exists():
    print(f'Filtrando carreteras de {PBF_PATH.name} con osmium (~30 s)...')
    _highway_types = (
        'motorway,trunk,primary,secondary,tertiary,'
        'residential,unclassified,road,living_street,'
        'motorway_link,trunk_link,primary_link,secondary_link,tertiary_link'
    )
    subprocess.run([
        'osmium', 'tags-filter',
        str(PBF_PATH),
        f'w/highway={_highway_types}',
        # osmium incluye nodos referenciados por defecto (sin flag extra)
        '-o', str(OSM_BZ2),
        '-O',  # --overwrite
    ], check=True)
    print(f'Guardado: {OSM_BZ2}  ({OSM_BZ2.stat().st_size / 1e6:.1f} MB)')
else:
    print(f'Usando fichero ya filtrado: {OSM_BZ2}  ({OSM_BZ2.stat().st_size / 1e6:.1f} MB)')

print('Cargando red vial en osmnx...')
G_osm = ox.graph_from_xml(str(OSM_BZ2), simplify=True, retain_all=False)
print(f'Red OSM: {G_osm.number_of_nodes():,} nodos  {G_osm.number_of_edges():,} aristas')

In [ ]:
# ── Anclar cada sensor a su nodo OSM más cercano ─────────────────────────────
# ox.nearest_nodes(G, X=longitudes, Y=latitudes) → array de osm node_ids

def _haversine_m(lat1, lon1, lat2, lon2, R=6_371_000.0):
    phi1, phi2 = _math.radians(lat1), _math.radians(lat2)
    dlat = _math.radians(lat2 - lat1)
    dlon = _math.radians(lon2 - lon1)
    a = _math.sin(dlat/2)**2 + _math.cos(phi1)*_math.cos(phi2)*_math.sin(dlon/2)**2
    return R * 2 * _math.asin(_math.sqrt(a))

osm_nearest_arr = ox.nearest_nodes(
    G_osm,
    X=sensors['longitud'].values.astype(float),
    Y=sensors['latitud'].values.astype(float),
)

sensor_to_osm = {
    sensors.iloc[i]['id']: osm_nearest_arr[i]
    for i in range(len(sensors))
}

# ── Diagnóstico de calidad del anclaje ────────────────────────────────────────
snap_dist = []
for i, (_, row) in enumerate(sensors.iterrows()):
    nd = G_osm.nodes[osm_nearest_arr[i]]
    snap_dist.append(
        _haversine_m(float(row['latitud']), float(row['longitud']),
                     float(nd['y']), float(nd['x']))
    )

snap_arr = np.array(snap_dist)
print(f'Sensores anclados    : {len(sensor_to_osm)}')
print(f'Snap máximo          : {snap_arr.max():.1f} m')
print(f'Snap mediana         : {np.median(snap_arr):.1f} m')
print(f'Sensores a >100 m    : {(snap_arr > 100).sum()}')
print(f'Sensores a >500 m    : {(snap_arr > 500).sum()}')
print('(Snap elevado indica sensor fuera de la red → considerar aumentar MARGEN_DEG)')


In [ ]:
# ── Función: construir grafo de sensores sobre red vial OSM ──────────────────
#
# Algoritmo:
#  Para cada sensor i del sentido:
#   1. Dijkstra desde osm_anchor[i] → road_dist a TODOS los sensores del sentido
#   2. Candidatos: sensores alcanzables con road_dist >= MIN_DIST_M (descarta <30m)
#      Ampliado con sensores lejanos si el BallTree haversine no captura suficientes
#   3. Ordena candidatos por road_dist ascendente
#   4. Itera candidatos; para cada j:
#        ¿Existe vecino k ya aceptado con road_dist(i,k)+road_dist(k,j) ≤ d(i,j)*TOL?
#        → sí: k es sensor intermedio, j se descarta (i→k ya fue añadido)
#        → no: añade i→j
#   5. Para al llegar a K vecinos directos aceptados

_K_OSM    = 3       # vecinos directos por sensor
_K_CAND   = 25      # candidatos haversine para pre-filtrado rápido
_CUTOFF_M = 25_000  # radio Dijkstra [m]; cubre cualquier salto real en Madrid
_TOL_INT  = 1.05    # tolerancia desigualdad triangular (5% detour)

def build_osm_sensor_graph(
    sensor_ids,
    sensor_to_osm,
    G_osm,
    sensors_df,
    K          = _K_OSM,
    min_dist_m = MIN_DIST_M,
    K_cand     = _K_CAND,
    cutoff_m   = _CUTOFF_M,
    tol        = _TOL_INT,
):
    """
    Construye DiGraph de sensores usando distancias viales reales (OSMnx/Dijkstra).

    Retorna (H, road_cache):
      H          : nx.DiGraph con atributo 'distancia_m' [m] por arista
      road_cache : dict {src: {tgt: road_dist_m}} — reutilizable para el mapa
    """
    sensor_list = sorted(sensor_ids)
    n           = len(sensor_list)
    df_sub      = sensors_df[sensors_df['id'].isin(sensor_list)].set_index('id')

    lats = np.array([float(df_sub.loc[s,'latitud'])  for s in sensor_list])
    lons = np.array([float(df_sub.loc[s,'longitud']) for s in sensor_list])
    bt   = BallTree(np.radians(np.column_stack([lats, lons])), metric='haversine')
    k_q  = min(K_cand + 1, n)
    _, idx_q = bt.query(np.radians(np.column_stack([lats, lons])), k=k_q)

    # ── Paso 1: Dijkstra desde cada sensor ────────────────────────────────────
    road_cache = {s: {} for s in sensor_list}
    osm_set    = {sensor_to_osm[s] for s in sensor_list}  # nodos OSM del sentido

    print(f'  Dijkstra para {n} sensores (cutoff={cutoff_m/1000:.0f} km)...')
    for i_s, src in enumerate(sensor_list):
        osm_src = sensor_to_osm[src]
        try:
            lengths = dict(nx.single_source_dijkstra_path_length(
                G_osm, osm_src, weight='length', cutoff=cutoff_m
            ))
        except Exception:
            lengths = {}
        for tgt in sensor_list:
            if tgt == src:
                continue
            osm_tgt = sensor_to_osm[tgt]
            if osm_tgt in lengths:
                road_cache[src][tgt] = lengths[osm_tgt]
        if (i_s + 1) % 50 == 0 or i_s + 1 == n:
            print(f'    {i_s+1}/{n}')

    # ── Paso 2: Construir grafo con filtro de sensor intermedio ──────────────
    H = nx.DiGraph()
    for sid in sensor_list:
        row = df_sub.loc[sid]
        H.add_node(sid,
            lat    = float(row['latitud']),
            lon    = float(row['longitud']),
            nombre = str(row['nombre']),
        )

    n_aristas = 0
    n_interm  = 0

    for i_s, src in enumerate(sensor_list):
        # Candidatos haversine (excluye self en col 0)
        haversine_cands = [sensor_list[idx_q[i_s][k]] for k in range(1, k_q)]

        # Candidatos válidos con road_dist disponible y >= min_dist_m
        cands = [
            (t, road_cache[src][t])
            for t in haversine_cands
            if t in road_cache[src] and road_cache[src][t] >= min_dist_m
        ]

        # Si hay pocos candidatos haversine válidos, añadir el resto del sentido
        if len(cands) < K:
            ya = {c[0] for c in cands}
            extra = sorted(
                [(t, d) for t, d in road_cache[src].items()
                 if d >= min_dist_m and t not in ya],
                key=lambda x: x[1]
            )
            cands.extend(extra[:K_cand])

        cands.sort(key=lambda x: x[1])

        # Añadir K vecinos directos aplicando filtro de sensor intermedio
        vecinos = []
        for tgt, d_ij in cands:
            if len(vecinos) >= K:
                break
            # Comprobar desigualdad triangular respecto a vecinos ya aceptados
            intermedio = any(
                d_ik + road_cache[k].get(tgt, float('inf')) <= d_ij * tol
                for k, d_ik in vecinos
            )
            if intermedio:
                n_interm += 1
            else:
                vecinos.append((tgt, d_ij))
                H.add_edge(src, tgt, distancia_m=round(d_ij, 1))
                n_aristas += 1

    print(f'  Aristas añadidas          : {n_aristas}')
    print(f'  Pares descartados (interm): {n_interm}')
    return H, road_cache


In [ ]:
# ── Construir subgrafos OSM para cada sentido ────────────────────────────────
# nodos_A (azul) y nodos_B (rojo) definidos en §11.
# naranja (sentido único, sin pareja) no se incluye — no tienen info de dirección.

print('═' * 50)
print('Sentido A (azul) — sensores:', len(nodos_A))
print('═' * 50)
G_A_osm, road_cache_A = build_osm_sensor_graph(
    nodos_A, sensor_to_osm, G_osm, sensors
)

print()
print('═' * 50)
print('Sentido B (rojo) — sensores:', len(nodos_B))
print('═' * 50)
G_B_osm, road_cache_B = build_osm_sensor_graph(
    nodos_B, sensor_to_osm, G_osm, sensors
)

print()
print('Construcción OSMnx completada.')
print(f'  G_A_osm: {G_A_osm.number_of_nodes()} nodos  {G_A_osm.number_of_edges()} aristas')
print(f'  G_B_osm: {G_B_osm.number_of_nodes()} nodos  {G_B_osm.number_of_edges()} aristas')


In [ ]:
# ── Garantizar conectividad fuerte en G_A_osm y G_B_osm ──────────────────────
# Reutiliza _hacer_fuertemente_conexo definida en §11.
# Las aristas puente añadidas aquí son por haversine (fallback) — son raras
# si el grafo OSM cubre bien la zona.

G_A_osm = _hacer_fuertemente_conexo(G_A_osm, 'G_A_osm', crd_g, id2idx_g, ids_g)
G_B_osm = _hacer_fuertemente_conexo(G_B_osm, 'G_B_osm', crd_g, id2idx_g, ids_g)

# ── Comparativa haversine vs OSMnx ───────────────────────────────────────────
def _stats(H):
    dists = [d['distancia_m'] for _, _, d in H.edges(data=True)]
    return {
        'nodos'      : H.number_of_nodes(),
        'aristas'    : H.number_of_edges(),
        'fuerte_cx'  : nx.is_strongly_connected(H),
        'dist_media' : round(float(np.mean(dists))) if dists else 0,
        'dist_max'   : round(max(dists)) if dists else 0,
    }

rows = [
    ('G_A  haversine (§11)', _stats(G_A)),
    ('G_A  OSMnx (§12)',     _stats(G_A_osm)),
    ('G_B  haversine (§11)', _stats(G_B)),
    ('G_B  OSMnx (§12)',     _stats(G_B_osm)),
]

print()
print(f'  {"Grafo":<24s} {"Nodos":>6s} {"Aristas":>8s} {"Fuerte":>7s} {"d_med [m]":>10s} {"d_max [m]":>10s}')
print('  ' + '-'*68)
for lbl, st in rows:
    print(f'  {lbl:<24s} {st["nodos"]:>6d} {st["aristas"]:>8d} '
          f'  {str(st["fuerte_cx"]):>5s}   {st["dist_media"]:>9d}  {st["dist_max"]:>9d}')


In [ ]:
# ── Mapa Folium con polilíneas de ruta real sobre la carretera ───────────────
# Para cada arista del grafo OSM calcula el camino más corto entre los dos
# nodos ancla en G_osm y dibuja la polilínea real (no línea recta).
# Si no existe ruta en OSM, usa línea recta punteada como fallback.

def _draw_osm_routes(H, G_osm, s2osm, fmap, color, label):
    """Dibuja las aristas de H como rutas reales sobre G_osm en fmap."""
    n_real, n_fb = 0, 0
    for u, v, data in H.edges(data=True):
        ou, ov = s2osm.get(u), s2osm.get(v)
        if ou is None or ov is None:
            n_fb += 1
            continue
        try:
            path_nds   = nx.shortest_path(G_osm, ou, ov, weight='length')
            path_coords = [(G_osm.nodes[nd]['y'], G_osm.nodes[nd]['x'])
                           for nd in path_nds]
            folium.PolyLine(
                path_coords, weight=2.5, color=color, opacity=0.80,
                tooltip=f'Sentido {label}: {u}\u2192{v} | {data["distancia_m"]:.0f} m',
            ).add_to(fmap)
            # Flecha al 65% de la ruta
            mid_i = max(0, int(len(path_coords) * 0.65) - 1)
            if mid_i + 1 < len(path_coords):
                p1, p2 = path_coords[mid_i], path_coords[mid_i + 1]
                brng = _bearing(p1[0], p1[1], p2[0], p2[1])
                folium.Marker(
                    location=p1,
                    icon=folium.DivIcon(
                        html=(f'<div style="transform:rotate({brng}deg);font-size:9px;'
                              f'color:{color};line-height:1;font-weight:bold;">'
                              f'&#9650;</div>'),
                        icon_size=(10, 10), icon_anchor=(5, 5),
                    ),
                ).add_to(fmap)
            n_real += 1
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            nu, nv = H.nodes[u], H.nodes[v]
            folium.PolyLine(
                [(nu['lat'], nu['lon']), (nv['lat'], nv['lon'])],
                weight=1.5, color=color, opacity=0.35, dash_array='8 4',
                tooltip=f'Fallback recta: {u}\u2192{v}',
            ).add_to(fmap)
            n_fb += 1
    print(f'  Sentido {label}: {n_real} rutas OSM reales | {n_fb} fallback línea recta')

m_osm = folium.Map(location=[lat_c, lon_c], zoom_start=13, tiles='CartoDB positron')

_COL_A = '#1565C0'
_COL_B = '#B71C1C'

print('Calculando rutas reales para el mapa...')
_draw_osm_routes(G_A_osm, G_osm, sensor_to_osm, m_osm, _COL_A, 'A')
_draw_osm_routes(G_B_osm, G_osm, sensor_to_osm, m_osm, _COL_B, 'B')

# Nodos como marcadores circulares
for H, color in [(G_A_osm, _COL_A), (G_B_osm, _COL_B)]:
    for node, data in H.nodes(data=True):
        folium.CircleMarker(
            location=[data['lat'], data['lon']],
            radius=5, color=color,
            fill=True, fill_color=color, fill_opacity=0.9,
            popup=folium.Popup(
                f'<b>{node}</b><br>{data["nombre"]}<br>'
                f'({data["lat"]:.5f}, {data["lon"]:.5f})<br>'
                f'Out: {H.out_degree(node)}  In: {H.in_degree(node)}',
                max_width=240,
            ),
        ).add_to(m_osm)

legend_osm = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:10px;border-radius:8px;
     border:1px solid #ccc;font-size:13px;">
  <b>Grafo OSMnx &mdash; rutas reales sobre carretera</b><br>
  <span style="color:#1565C0;font-size:16px;">&#9679;</span> Sentido A (ida / azul)<br>
  <span style="color:#B71C1C;font-size:16px;">&#9679;</span> Sentido B (vuelta / rojo)<br>
  <hr style="margin:4px 0;">
  <span style="font-size:10px;">&#x2015; ruta real OSM &nbsp;&nbsp;
  &#x2015; &#x2015; fallback recta</span>
</div>
"""
m_osm.get_root().html.add_child(folium.Element(legend_osm))

out_osm_html = OUTPUT_DIR / 'grafo_osm_rutas_reales.html'
m_osm.save(str(out_osm_html))
print(f'Mapa guardado: {out_osm_html}')
m_osm


In [ ]:
# ── Guardar grafos OSMnx como GraphML ────────────────────────────────────────
# Sobreescribe los archivos generados en §11 (haversine) con la versión OSMnx.

nx.write_graphml(G_A_osm, OUTPUT_DIR / 'grafo_sentido_A.graphml')
nx.write_graphml(G_B_osm, OUTPUT_DIR / 'grafo_sentido_B.graphml')

print(f'Guardado: {OUTPUT_DIR}/grafo_sentido_A.graphml'
      f'  ({G_A_osm.number_of_nodes()} nodos, {G_A_osm.number_of_edges()} aristas)')
print(f'Guardado: {OUTPUT_DIR}/grafo_sentido_B.graphml'
      f'  ({G_B_osm.number_of_nodes()} nodos, {G_B_osm.number_of_edges()} aristas)')
print()
# Comparativa final haversine § 11 vs OSMnx §12
print('Diferencia haversine (§11) vs OSMnx (§12):')
for lbl_h, H_h, lbl_o, H_o in [
    ('G_A §11', G_A, 'G_A §12', G_A_osm),
    ('G_B §11', G_B, 'G_B §12', G_B_osm),
]:
    delta_a = H_o.number_of_edges() - H_h.number_of_edges()
    print(f'  {lbl_h}: {H_h.number_of_edges()} aristas  →  '
          f'{lbl_o}: {H_o.number_of_edges()} aristas  (Δ={delta_a:+d})')
print()
print('Los grafos OSMnx ya están listos para Dijkstra con tiempos de viaje reales.')


## 13. Subgrafos por tipo de elemento: URB y M30

Separa los sensores en dos DiGraph independientes según `tipo_elem`:
- **G_URB** — 985 sensores urbanos (intersecciones, avenidas, calles)
- **G_M30** — 84 sensores de la M-30 (autopista de circunvalación)

Cada subgrafo hereda las aristas de **G** (§3) que unen sensores del mismo tipo y
se garantiza conectividad fuerte con `_hacer_fuertemente_conexo` (§11).

Ficheros generados:
- `outputs/grafo_tipo_URB.graphml`
- `outputs/grafo_tipo_M30.graphml`
- `outputs/grafo_tipos_URB_M30.html`


In [ ]:
# ── Partición de nodos por tipo_elem ─────────────────────────────────────────
sensor_tipo = sensors.set_index('id')['tipo_elem'].to_dict()

nodos_URB = {n for n in G.nodes() if sensor_tipo.get(n) == 'URB'}
nodos_M30 = {n for n in G.nodes() if sensor_tipo.get(n) == 'M30'}

print(f'Nodos URB : {len(nodos_URB)}')
print(f'Nodos M30 : {len(nodos_M30)}')
print()

# ── Subgrafos inducidos (aristas solo entre nodos del mismo tipo) ─────────────
G_URB = G.subgraph(nodos_URB).copy()
G_M30 = G.subgraph(nodos_M30).copy()

print(f'G_URB inducido → nodos: {G_URB.number_of_nodes():>4}  '
      f'aristas: {G_URB.number_of_edges():>5}  '
      f'WCC: {nx.number_weakly_connected_components(G_URB)}')
print(f'G_M30 inducido → nodos: {G_M30.number_of_nodes():>4}  '
      f'aristas: {G_M30.number_of_edges():>5}  '
      f'WCC: {nx.number_weakly_connected_components(G_M30)}')
print()

# ── Garantizar conectividad fuerte ────────────────────────────────────────────
G_URB = _hacer_fuertemente_conexo(G_URB, 'G_URB', crd_g, id2idx_g, ids_g)
G_M30 = _hacer_fuertemente_conexo(G_M30, 'G_M30', crd_g, id2idx_g, ids_g)

# ── Estadísticas ──────────────────────────────────────────────────────────────
sep = '-' * 60
print()
print(sep)
print(f"  {'Métrica':<28s}  {'G_URB':>10s}  {'G_M30':>10s}")
print(sep)

for etiqueta, H in [('G_URB', G_URB), ('G_M30', G_M30)]:
    dists    = [d['distancia_m'] for _, _, d in H.edges(data=True)]
    out_degs = [H.out_degree(n) for n in H.nodes()]
    locals()[f'_st_{etiqueta}'] = {
        'Nodos'              : H.number_of_nodes(),
        'Aristas'            : H.number_of_edges(),
        'Fuertemente conexo' : nx.is_strongly_connected(H),
        'Grado salida medio' : round(float(np.mean(out_degs)), 2),
        'Dist media (m)'     : round(float(np.mean(dists))),
        'Dist máxima (m)'    : round(max(dists)),
        'Dist mínima (m)'    : round(min(dists)),
    }

for metrica in _st_G_URB:
    print(f"  {metrica:<28s}  {str(_st_G_URB[metrica]):>10s}  {str(_st_G_M30[metrica]):>10s}")
print(sep)

# ── Guardar ───────────────────────────────────────────────────────────────────
nx.write_graphml(G_URB, OUTPUT_DIR / 'grafo_tipo_URB.graphml')
nx.write_graphml(G_M30, OUTPUT_DIR / 'grafo_tipo_M30.graphml')
print(f'\nGuardado: {OUTPUT_DIR}/grafo_tipo_URB.graphml')
print(f'Guardado: {OUTPUT_DIR}/grafo_tipo_M30.graphml')


In [ ]:
# ── Mapa Folium: G_URB (verde) + G_M30 (morado) ──────────────────────────────
_COL_URB = '#2E7D32'   # verde oscuro
_COL_M30 = '#6A1B9A'   # morado oscuro

m_tipo = folium.Map(location=[lat_c, lon_c], zoom_start=13, tiles='CartoDB positron')

for H, color, lbl in [(G_URB, _COL_URB, 'URB'), (G_M30, _COL_M30, 'M30')]:
    for u, v, data in H.edges(data=True):
        nu, nv = H.nodes[u], H.nodes[v]
        p1 = (nu['lat'], nu['lon'])
        p2 = (nv['lat'], nv['lon'])
        folium.PolyLine(
            [p1, p2], weight=1.5, color=color, opacity=0.55,
            tooltip=f'{lbl}: {u}→{v} | {data["distancia_m"]:.0f} m',
        ).add_to(m_tipo)
        mid_lat = p1[0] + 0.65 * (p2[0] - p1[0])
        mid_lon = p1[1] + 0.65 * (p2[1] - p1[1])
        brng = _bearing(p1[0], p1[1], p2[0], p2[1])
        folium.Marker(
            location=(mid_lat, mid_lon),
            icon=folium.DivIcon(
                html=(f'<div style="transform:rotate({brng}deg);font-size:7px;'
                      f'color:{color};line-height:1;">▲</div>'),
                icon_size=(8, 8), icon_anchor=(4, 4),
            ),
        ).add_to(m_tipo)

for H, color in [(G_URB, _COL_URB), (G_M30, _COL_M30)]:
    for node, data in H.nodes(data=True):
        folium.CircleMarker(
            location=[data['lat'], data['lon']],
            radius=4 if color == _COL_URB else 6,
            color=color, fill=True, fill_color=color, fill_opacity=0.9,
            popup=folium.Popup(
                f'<b>{node}</b><br>{data["nombre"]}<br>'
                f'({data["lat"]:.5f}, {data["lon"]:.5f})<br>'
                f'Out: {H.out_degree(node)}  In: {H.in_degree(node)}',
                max_width=240,
            ),
        ).add_to(m_tipo)

legend_tipo = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:10px;border-radius:8px;
     border:1px solid #ccc;font-size:13px;">
  <b>Subgrafos por tipo de elemento</b><br>
  <span style="color:#2E7D32;font-size:16px;">&#9679;</span> URB (urbano, 985 sensores)<br>
  <span style="color:#6A1B9A;font-size:16px;">&#9679;</span> M30 (autopista, 84 sensores)<br>
  <span style="font-size:10px;">&#9650; dirección del flujo</span>
</div>
"""
m_tipo.get_root().html.add_child(folium.Element(legend_tipo))

out_tipo = OUTPUT_DIR / 'grafo_tipos_URB_M30.html'
m_tipo.save(str(out_tipo))
print(f'Mapa guardado: {out_tipo}')
m_tipo
